# VideoMAE Fine-Tuning Pipeline

Fine-tunes **VideoMAE-Base** (Masked Autoencoder pre-trained on Kinetics-400) 
on the driving distraction dataset.

**Why VideoMAE over TimeSformer?**
- Pre-trained with 90-95% tube masking → much more data-efficient on small datasets
- Uses 224×224 (vs 448×448) → ~2× lower VRAM → batch=2 on P100/T4
- Same HuggingFace Trainer API

---
### Kaggle setup
1. **Add-ons → Secrets** → add `HF_TOKEN` with **write** scope.
2. Enable **Internet** and **GPU** (P100 recommended).
3. Run all cells top to bottom.

## 1. Detect Environment & Set Paths

In [ ]:
# ── Detect environment & set default paths ─────────────────────────────────────
if ON_KAGGLE:
    PLATFORM = 'kaggle'
    REPO_DIR = '/kaggle/working/Driving_Distraction_Detection'
    HF_CACHE_DIR = '/kaggle/working/hf_cache'      # default – will be overridden below
    OUTPUT_DIR   = '/kaggle/working/videomae_outputs'  # default – will be overridden below
elif ON_COLAB:
    PLATFORM = 'colab'
    REPO_DIR = '/content/Driving_Distraction_Detection'
    HF_CACHE_DIR = '/content/hf_cache'
    OUTPUT_DIR   = '/content/drive/MyDrive/VideoMAE_Outputs'
else:
    PLATFORM = 'local'
    REPO_DIR = os.getcwd()
    HF_CACHE_DIR = './hf_cache'
    OUTPUT_DIR   = './videomae_outputs'

print(f'Platform:{PLATFORM}  Cache:{HF_CACHE_DIR}  Output:{OUTPUT_DIR}')

# --------------------------------------------------------------------------- #
#    **Override Kaggle paths** – keep outputs in `/kaggle/working` but move the
#    Hugging Face cache to the large temporary space `/tmp`.
# --------------------------------------------------------------------------- #
if PLATFORM == 'kaggle':
    HF_CACHE_DIR = '/tmp/hf_cache'                 # Cache for HuggingFace assets
    OUTPUT_DIR   = '/kaggle/working/videomae_outputs'  # Training outputs, checkpoints, logs
    print(f'Adjusted paths → HF_CACHE_DIR: {HF_CACHE_DIR}, OUTPUT_DIR: {OUTPUT_DIR}')


Platform:kaggle  Cache:/kaggle/working/hf_cache  Output:/kaggle/working/videomae_outputs
Adjusted paths → HF_CACHE_DIR: /tmp/hf_cache, OUTPUT_DIR: /tmp/videomae_outputs


## 2. (Colab only) Mount Google Drive

In [ ]:
if PLATFORM=='colab':
    from google.colab import drive; drive.mount('/content/drive'); print('Mounted.')
else: print(f'Skip ({PLATFORM})')

## 3. Clone Repository

In [5]:
if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git {REPO_DIR}')
else: os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR); print(f'CWD: {os.getcwd()}')

Cloning into '/kaggle/working/Driving_Distraction_Detection'...


CWD: /kaggle/working/Driving_Distraction_Detection


## 4. Install Dependencies

In [6]:
# Filter out simpleaudio on Kaggle/Colab since it's headless and simpleaudio fails to build (requires ALSA development headers)
req_file = 'requirements.txt'
if ON_KAGGLE or ON_COLAB:
    req_file = 'requirements_headless.txt'
    with open('requirements.txt', 'r') as f:
        lines = f.readlines()
    with open(req_file, 'w') as f:
        for line in lines:
            if 'simpleaudio' not in line:
                f.write(line)
    print('Filtered requirements.txt -> requirements_headless.txt (removed simpleaudio)')

!pip install -q -r {req_file}
!pip install -q accelerate -U
print('Done.')

Filtered requirements.txt -> requirements_headless.txt (removed simpleaudio)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 63.7 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.4 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━

## 5. Hugging Face Authentication

In [7]:
from huggingface_hub import login
if PLATFORM=='kaggle':
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('Logged in via Kaggle Secret.')
else:
    hf_token = os.environ.get('HF_TOKEN','')
    if hf_token: login(token=hf_token, add_to_git_credential=False); print('Logged in via env.')
    else:
        from huggingface_hub import notebook_login; notebook_login()

Logged in via Kaggle Secret.


## 6. Configuration

| Variable | Options / Description |
|---|---|
| `MODEL_VARIANT` | `'kinetics'` (general actions) or `'ssv2'` (hand/object interactions) |
| `VIDEOS_PER_CLASS` | clips to download (None = all ~13 GB) |
| `HF_REPO_ID` | `'username/repo'` to push checkpoints to HF Hub, or `''` to save locally |

In [8]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
MODEL_VARIANT    = 'kinetics'  # 'kinetics' or 'ssv2'
# Custom per-class limits (int or comma-separated name:limit string)
# Set talking_to_passenger:0 as it is not used in videomae_finetuning
#VIDEOS_PER_CLASS = 'safe_driving:179,texting_right:239,phonecall_right:168,texting_left:213,phonecall_left:179,radio:147,drinking:250,reach_side:200,hair_and_makeup:266,change_gear:250,talking_to_passenger:0'
VIDEOS_PER_CLASS = 'safe_driving:10,texting_right:10,phonecall_right:10,texting_left:10,phonecall_left:10,radio:10,drinking:10,reach_side:10,hair_and_makeup:10,change_gear:10,talking_to_passenger:0'

DOWNLOAD_SEED    = 42
HF_REPO_ID       = ''          # e.g. 'your_username/videomae-distraction'
USE_WANDB        = True
# ──────────────────────────────────────────────────────────────────────────────
MODEL_ID_MAP = {
    'kinetics': 'MCG-NJU/videomae-base-finetuned-kinetics',
    'ssv2':     'MCG-NJU/videomae-base-finetuned-ssv2',
}
MODEL_HF_ID = MODEL_ID_MAP.get(MODEL_VARIANT, MODEL_ID_MAP['kinetics'])
print(f'Model  : {MODEL_VARIANT} → {MODEL_HF_ID}')
print(f'Videos : {VIDEOS_PER_CLASS}')
print(f'HF Hub : {HF_REPO_ID or "(disabled)"}')

Model  : kinetics → MCG-NJU/videomae-base-finetuned-kinetics
Videos : safe_driving:10,texting_right:10,phonecall_right:10,texting_left:10,phonecall_left:10,radio:10,drinking:10,reach_side:10,hair_and_makeup:10,change_gear:10,talking_to_passenger:0
HF Hub : (disabled)


## 7. Download Model & Dataset

In [9]:
# Il modello VideoMAE viene scaricato automaticamente da HF Hub durante il training.
# Qui scaricamo solo il dataset video.
cmd = f'python scripts/download_assets.py --output_dir {HF_CACHE_DIR} --seed {DOWNLOAD_SEED}'
if VIDEOS_PER_CLASS: cmd += f' --videos_per_class {VIDEOS_PER_CLASS}'
print(f'Running: {cmd}\n')
!{cmd}

Running: python scripts/download_assets.py --output_dir /tmp/hf_cache --seed 42 --videos_per_class safe_driving:10,texting_right:10,phonecall_right:10,texting_left:10,phonecall_left:10,radio:10,drinking:10,reach_side:10,hair_and_makeup:10,change_gear:10,talking_to_passenger:0

Storage root: /tmp/hf_cache

[1/2] Downloading model 'MCG-NJU/videomae-base-finetuned-kinetics' -> /tmp/hf_cache/videomae-base
      Model saved to: /tmp/hf_cache/videomae-base

[2/2] Listing files in dataset repo 'endoard/distraction_detection_dataset'...
  safe_driving             :   10 / 179 videos selected
  texting_right            :   10 / 239 videos selected
  phonecall_right          :   10 / 168 videos selected
  texting_left             :   10 / 213 videos selected
  phonecall_left           :   10 / 179 videos selected
  radio                    :   10 / 147 videos selected
  drinking                 :   10 / 298 videos selected
  reach_side               :   10 / 484 videos selected
  hair_and_makeup

## 8. Start Training

In [10]:
# MODEL_PATH = HF Hub model ID → scaricato/cachato automaticamente da HuggingFace
os.environ['MODEL_PATH']       = MODEL_HF_ID
os.environ['DATASET_PATH']     = os.path.join(HF_CACHE_DIR, 'distraction_dataset')
os.environ['OUTPUT_DIR']       = OUTPUT_DIR
os.environ['HF_REPO_ID']       = HF_REPO_ID
os.environ['TRAIN_BATCH_SIZE'] = '2'   # P100 gestisce batch=2 a 224x224
os.environ['EVAL_BATCH_SIZE']  = '2'
os.environ['GRAD_ACCUM_STEPS'] = '8'   # effective batch = 16
if VIDEOS_PER_CLASS: os.environ['LIMIT_CAP'] = str(VIDEOS_PER_CLASS)

if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        wkey = UserSecretsClient().get_secret('WANDB_API_KEY')
        if wkey:
            os.environ['WANDB_API_KEY'] = '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'
            print('Loaded WANDB_API_KEY from Kaggle Secrets.')
    except Exception:
        pass
    import wandb
    try:
        wandb.login(key=os.environ.get('WANDB_API_KEY'))
        print('WandB login successful.')
    except Exception as e:
        print(f'WandB login prompt: {e}')
else:
    os.environ['WANDB_MODE']='disabled'; os.environ['WANDB_DISABLED']='true'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'MODEL_PATH   = {os.environ["MODEL_PATH"]}')
print(f'DATASET_PATH = {os.environ["DATASET_PATH"]}')
print(f'OUTPUT_DIR   = {os.environ["OUTPUT_DIR"]}')
print(f'HF_REPO_ID   = {os.environ["HF_REPO_ID"] or "(disabled)"}')
print()
!python scripts/videomae_finetuning.py

Loaded WANDB_API_KEY from Kaggle Secrets.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ensoli-1918623 (ensoli-1918623-la-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB login successful.
MODEL_PATH   = MCG-NJU/videomae-base-finetuned-kinetics
DATASET_PATH = /tmp/hf_cache/distraction_dataset
OUTPUT_DIR   = /tmp/videomae_outputs
HF_REPO_ID   = (disabled)

Model      : MCG-NJU/videomae-base-finetuned-kinetics
Dataset    : /tmp/hf_cache/distraction_dataset
Output     : /tmp/videomae_outputs
Frames     : 16
Limit cap  : {'safe_driving': 10, 'texting_right': 10, 'phonecall_right': 10, 'texting_left': 10, 'phonecall_left': 10, 'radio': 10, 'drinking': 10, 'reach_side': 10, 'hair_and_makeup': 10, 'change_gear': 10, 'talking_to_passenger': 0}
HF Repo    : (disabled — saving locally only)

preprocessor_config.json: 100%|████████████████| 271/271 [00:00<00:00, 1.31MB/s]
config.json: 22.9kB [00:00, 39.2MB/s]
model.safetensors: 100%|██████████████████████| 346M/346M [00:03<00:00, 114MB/s]
Loading weights: 100%|█| 186/186 [00:00<00:00, 1526.39it/s, Materializing param=
VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
K

## 9. Inspect Outputs

In [ ]:
if os.path.exists(OUTPUT_DIR):
    for root, dirs, files in os.walk(OUTPUT_DIR):
        lvl = root.replace(OUTPUT_DIR,'').count(os.sep)
        pad = '  '*lvl
        print(f'{pad}{os.path.basename(root)}/')
        for f in files:
            mb = os.path.getsize(os.path.join(root,f))/1024**2
            print(f'{pad}  {f}  ({mb:.1f} MB)')
else: print('Output dir not found.')